#E-Commerce Sales Analytics Dashboard

##Analisis Performa Penjualan, Produk, Customer, dan Kepuasan Pelanggan

In [ ]:
!pip install -q pandas numpy matplotlib seaborn plotly streamlit pyngrok

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving archive (23).zip to archive (23).zip


In [ ]:
import os

print(os.listdir())

['.config', 'archive (23).zip', 'sample_data']


In [ ]:
!unzip -o archive.zip -d olist_data

unzip:  cannot find or open archive.zip, archive.zip.zip or archive.zip.ZIP.


In [ ]:
!unzip -o brazilian-ecommerce.zip -d olist_data

unzip:  cannot find or open brazilian-ecommerce.zip, brazilian-ecommerce.zip.zip or brazilian-ecommerce.zip.ZIP.


In [ ]:
!unzip -o archive.zip -d olist_data

unzip:  cannot find or open archive.zip, archive.zip.zip or archive.zip.ZIP.


In [ ]:
import os

print(os.listdir())

['.config', 'olist_data', 'archive (23).zip', 'sample_data']


In [ ]:
!mkdir -p olist_data

In [ ]:
!unzip -o "archive (23).zip" -d olist_data

Archive:  archive (23).zip
  inflating: olist_data/olist_customers_dataset.csv  
  inflating: olist_data/olist_geolocation_dataset.csv  
  inflating: olist_data/olist_order_items_dataset.csv  
  inflating: olist_data/olist_order_payments_dataset.csv  
  inflating: olist_data/olist_order_reviews_dataset.csv  
  inflating: olist_data/olist_orders_dataset.csv  
  inflating: olist_data/olist_products_dataset.csv  
  inflating: olist_data/olist_sellers_dataset.csv  
  inflating: olist_data/product_category_name_translation.csv  


In [ ]:
import os

print(os.listdir("olist_data"))

['olist_orders_dataset.csv', 'olist_customers_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_items_dataset.csv', 'product_category_name_translation.csv', 'olist_products_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_sellers_dataset.csv', 'olist_geolocation_dataset.csv']


In [ ]:
!pip install -q pandas numpy matplotlib seaborn plotly streamlit pyngrok

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display
import os
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("✅ Semua library berhasil di-load!")

✅ Semua library berhasil di-load!


In [ ]:
DATA_PATH = "olist_data"

print("📁 Isi folder dataset:")
print(os.listdir(DATA_PATH))

📁 Isi folder dataset:
['olist_orders_dataset.csv', 'olist_customers_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_items_dataset.csv', 'product_category_name_translation.csv', 'olist_products_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_sellers_dataset.csv', 'olist_geolocation_dataset.csv']


In [ ]:
customers = pd.read_csv(
    f"{DATA_PATH}/olist_customers_dataset.csv"
)

geolocation = pd.read_csv(
    f"{DATA_PATH}/olist_geolocation_dataset.csv"
)

order_items = pd.read_csv(
    f"{DATA_PATH}/olist_order_items_dataset.csv"
)

payments = pd.read_csv(
    f"{DATA_PATH}/olist_order_payments_dataset.csv"
)

reviews = pd.read_csv(
    f"{DATA_PATH}/olist_order_reviews_dataset.csv"
)

orders = pd.read_csv(
    f"{DATA_PATH}/olist_orders_dataset.csv"
)

products = pd.read_csv(
    f"{DATA_PATH}/olist_products_dataset.csv"
)

sellers = pd.read_csv(
    f"{DATA_PATH}/olist_sellers_dataset.csv"
)

translation = pd.read_csv(
    f"{DATA_PATH}/product_category_name_translation.csv"
)

print("✅ Semua dataset berhasil dimuat!")

✅ Semua dataset berhasil dimuat!


In [ ]:
datasets = {
    "Customers": customers,
    "Geolocation": geolocation,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Orders": orders,
    "Products": products,
    "Sellers": sellers,
    "Translation": translation
}

overview = []

for name, data in datasets.items():
    overview.append({
        "Dataset": name,
        "Rows": data.shape[0],
        "Columns": data.shape[1],
        "Missing Values": data.isnull().sum().sum(),
        "Duplicate Rows": data.duplicated().sum()
    })

overview_df = pd.DataFrame(overview)

display(overview_df)

,Dataset,Rows,Columns,Missing Values,Duplicate Rows
0,Customers,99441,5,0,0
1,Geolocation,1000163,5,0,261831
2,Order Items,112650,7,0,0
3,Payments,103886,5,0,0
4,Reviews,99224,7,145903,0
5,Orders,99441,8,4908,0
6,Products,32951,9,2448,0
7,Sellers,3095,4,0,0
8,Translation,71,2,0,0


In [ ]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [ ]:
missing_orders = (
    orders.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_orders = missing_orders[
    missing_orders > 0
]

display(missing_orders)

,0
order_delivered_customer_date,2965
order_delivered_carrier_date,1783
order_approved_at,160


In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

print("✅ Kolom tanggal berhasil dikonversi.")

✅ Kolom tanggal berhasil dikonversi.


In [ ]:
products["product_category_name"] = (
    products["product_category_name"]
    .fillna("unknown")
)

products["product_name_lenght"] = (
    products["product_name_lenght"]
    .fillna(products["product_name_lenght"].median())
)

products["product_description_lenght"] = (
    products["product_description_lenght"]
    .fillna(products["product_description_lenght"].median())
)

products["product_photos_qty"] = (
    products["product_photos_qty"]
    .fillna(0)
)

products["product_weight_g"] = (
    products["product_weight_g"]
    .fillna(products["product_weight_g"].median())
)

products["product_length_cm"] = (
    products["product_length_cm"]
    .fillna(products["product_length_cm"].median())
)

products["product_height_cm"] = (
    products["product_height_cm"]
    .fillna(products["product_height_cm"].median())
)

products["product_width_cm"] = (
    products["product_width_cm"]
    .fillna(products["product_width_cm"].median())
)

print("✅ Product data berhasil dibersihkan.")

✅ Product data berhasil dibersihkan.


In [ ]:
translation["product_category_name_english"] = (
    translation["product_category_name_english"]
    .fillna("unknown")
)

print(
    "Jumlah kategori:",
    translation["product_category_name_english"].nunique()
)

Jumlah kategori: 71


In [ ]:
review_summary = (
    reviews
    .groupby("order_id")
    .agg(
        review_score=("review_score", "mean")
    )
    .reset_index()
)

review_summary.head()

,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0


In [ ]:
review_summary = (
    reviews
    .groupby("order_id")
    .agg(
        review_score=("review_score", "mean")
    )
    .reset_index()
)

review_summary.head()

,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0


In [ ]:
payment_summary = (
    payments
    .groupby("order_id")
    .agg(
        payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max")
    )
    .reset_index()
)

payment_summary.head()

,order_id,payment_value,payment_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,3
2,000229ec398224ef6ca0657da4fc703e,216.87,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3


In [ ]:
products_en = products.merge(
    translation,
    on="product_category_name",
    how="left"
)

products_en["product_category_name_english"] = (
    products_en["product_category_name_english"]
    .fillna("Unknown")
)

products_en.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


In [ ]:
df = orders.merge(
    customers,
    on="customer_id",
    how="left"
)

df = df.merge(
    order_items,
    on="order_id",
    how="left"
)

df = df.merge(
    products_en,
    on="product_id",
    how="left"
)

df = df.merge(
    payment_summary,
    on="order_id",
    how="left"
)

df = df.merge(
    review_summary,
    on="order_id",
    how="left"
)

print("Ukuran dataset final:")
print(df.shape)

Ukuran dataset final:
(113425, 30)


In [ ]:
display(df.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,payment_value,payment_installments,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,38.71,1.0,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,141.46,1.0,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,179.12,3.0,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop,72.20,1.0,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery,28.62,1.0,5.0


In [ ]:
df["revenue"] = (
    df["price"].fillna(0)
    +
    df["freight_value"].fillna(0)
)

df["order_date"] = (
    df["order_purchase_timestamp"]
    .dt.date
)

df["order_year"] = (
    df["order_purchase_timestamp"]
    .dt.year
)

df["order_month"] = (
    df["order_purchase_timestamp"]
    .dt.month
)

df["month_name"] = (
    df["order_purchase_timestamp"]
    .dt.strftime("%b")
)

df["year_month"] = (
    df["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

df["weekday"] = (
    df["order_purchase_timestamp"]
    .dt.day_name()
)

print("✅ Feature engineering selesai.")

✅ Feature engineering selesai.


In [ ]:
df["delivery_days"] = (
    df["order_delivered_customer_date"]
    -
    df["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

df["estimated_delivery_days"] = (
    df["order_estimated_delivery_date"]
    -
    df["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

df["delivery_delay_days"] = (
    df["delivery_days"]
    -
    df["estimated_delivery_days"]
)

df["delivery_days"] = (
    df["delivery_days"]
    .clip(lower=0)
)

print("✅ Delivery features selesai.")

✅ Delivery features selesai.


In [ ]:
df[
    [
        "price",
        "freight_value",
        "revenue",
        "payment_value",
        "review_score",
        "delivery_days"
    ]
].describe()

,price,freight_value,revenue,payment_value,review_score,delivery_days
count,112650.000000,112650.000000,113425.000000,113422.000000,112464.000000,110196.000000
mean,120.653739,19.990320,139.683079,180.482857,4.017589,12.472631
std,183.633928,15.806405,190.424476,273.548227,1.398415,9.445655
min,0.850000,0.000000,0.000000,0.000000,1.000000,0.533414
25%,39.900000,13.080000,54.780000,65.680000,4.000000,6.736262
50%,74.990000,16.260000,91.500000,114.400000,5.000000,10.184282
75%,134.900000,21.150000,157.360000,195.360000,5.000000,15.541097
max,6735.000000,409.680000,6929.310000,13664.080000,5.000000,209.628611


In [ ]:
final_missing = (
    df.isnull()
    .sum()
    .sort_values(ascending=False)
)

final_missing = final_missing[
    final_missing > 0
]

display(final_missing.head(20))

,0
order_delivered_customer_date,3229
delivery_delay_days,3229
delivery_days,3229
order_delivered_carrier_date,1968
review_score,961
shipping_limit_date,775
seller_id,775
product_name_lenght,775
product_category_name_english,775
product_width_cm,775


In [ ]:
total_revenue = df["revenue"].sum()

total_orders = df["order_id"].nunique()

total_customers = df["customer_unique_id"].nunique()

total_products = df["product_id"].nunique()

average_order_value = (
    total_revenue / total_orders
)

average_rating = (
    df["review_score"].mean()
)

print("💰 Total Revenue       :", f"R$ {total_revenue:,.2f}")
print("🛒 Total Orders        :", f"{total_orders:,}")
print("👥 Total Customers     :", f"{total_customers:,}")
print("📦 Total Products      :", f"{total_products:,}")
print("💳 Average Order Value :", f"R$ {average_order_value:,.2f}")
print("⭐ Average Rating      :", f"{average_rating:.2f}/5")

💰 Total Revenue       : R$ 15,843,553.24
🛒 Total Orders        : 99,441
👥 Total Customers     : 96,096
📦 Total Products      : 32,951
💳 Average Order Value : R$ 159.33
⭐ Average Rating      : 4.02/5


In [ ]:
monthly_sales = (
    df.groupby("year_month")
    .agg(
        revenue=("revenue", "sum"),
        orders=("order_id", "nunique")
    )
    .reset_index()
)

monthly_sales.head()

,year_month,revenue,orders
0,2016-09,354.75,4
1,2016-10,56808.84,324
2,2016-12,19.62,1
3,2017-01,137188.49,800
4,2017-02,286280.62,1780


In [ ]:
fig = px.line(
    monthly_sales,
    x="year_month",
    y="revenue",
    markers=True,
    title="Monthly Revenue Trend"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Revenue"
)

fig.show()

In [ ]:
category_analysis = (
    df.groupby("product_category_name_english")
    .agg(
        revenue=("revenue", "sum"),
        orders=("order_id", "nunique"),
        quantity=("order_item_id", "count")
    )
    .sort_values(
        "revenue",
        ascending=False
    )
    .reset_index()
)

display(
    category_analysis.head(15)
)

,product_category_name_english,revenue,orders,quantity
0,health_beauty,1441248.07,8836,9670
1,watches_gifts,1305541.61,5624,5991
2,bed_bath_table,1241681.72,9417,11115
3,sports_leisure,1156656.48,7720,8641
4,computers_accessories,1059272.40,6689,7827
5,furniture_decor,902511.79,6449,8334
6,housewares,778397.77,5884,6964
7,cool_stuff,719329.95,3632,3796
8,auto,685384.32,3897,4235
9,garden_tools,584219.21,3518,4347


In [ ]:
top_categories = category_analysis.head(15)

fig = px.bar(
    top_categories.sort_values("revenue"),
    x="revenue",
    y="product_category_name_english",
    orientation="h",
    title="Top 15 Product Categories by Revenue"
)

fig.show()

In [ ]:
customer_analysis = (
    df.groupby("customer_unique_id")
    .agg(
        orders=("order_id", "nunique"),
        spending=("revenue", "sum")
    )
    .reset_index()
)

customer_analysis["customer_type"] = np.where(
    customer_analysis["orders"] > 1,
    "Repeat Customer",
    "One-Time Customer"
)

display(
    customer_analysis.head()
)

,customer_unique_id,orders,spending,customer_type
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,One-Time Customer
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,One-Time Customer
2,0000f46a3911fa3c0805444483337064,1,86.22,One-Time Customer
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,One-Time Customer
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,One-Time Customer


In [ ]:
customer_type = (
    customer_analysis["customer_type"]
    .value_counts()
    .reset_index()
)

customer_type.columns = [
    "customer_type",
    "customers"
]

display(customer_type)

,customer_type,customers
0,One-Time Customer,93099
1,Repeat Customer,2997


In [ ]:
fig = px.pie(
    customer_type,
    names="customer_type",
    values="customers",
    hole=0.5,
    title="Customer Type Distribution"
)

fig.show()

In [ ]:
state_analysis = (
    df.groupby("customer_state")
    .agg(
        customers=("customer_unique_id", "nunique"),
        orders=("order_id", "nunique"),
        revenue=("revenue", "sum")
    )
    .sort_values(
        "revenue",
        ascending=False
    )
    .reset_index()
)

display(state_analysis)

,customer_state,customers,orders,revenue
0,SP,40302,41746,5921678.12
1,RJ,12384,12852,2129681.98
2,MG,11259,11635,1856161.49
3,RS,5277,5466,885826.76
4,PR,4882,5045,800935.44
5,BA,3277,3380,611506.67
6,SC,3534,3637,610213.60
7,DF,2075,2140,353229.44
8,GO,1952,2020,347706.93
9,ES,1964,2033,324801.91


In [ ]:
fig = px.bar(
    state_analysis.head(15),
    x="customer_state",
    y="revenue",
    title="Top States by Revenue"
)

fig.show()

In [ ]:
review_analysis = (
    df.groupby("review_score")
    .agg(
        orders=("order_id", "nunique"),
        revenue=("revenue", "sum")
    )
    .reset_index()
)

display(review_analysis)

,review_score,orders,revenue
0,1.000000,11316,2098718.37
1,1.500000,8,1637.62
2,2.000000,3125,526409.87
3,2.500000,34,4396.51
4,3.000000,8136,1222166.86
5,3.333333,1,42.11
6,3.500000,25,3591.14
7,4.000000,19018,2940109.10
8,4.333333,1,42.77
9,4.500000,54,6582.05


In [ ]:
fig = px.bar(
    review_analysis,
    x="review_score",
    y="orders",
    title="Customer Review Score Distribution"
)

fig.show()

In [ ]:
delivery_analysis = (
    df.groupby("order_status")
    .agg(
        orders=("order_id", "nunique")
    )
    .reset_index()
)

display(delivery_analysis)

,order_status,orders
0,approved,2
1,canceled,625
2,created,5
3,delivered,96478
4,invoiced,314
5,processing,301
6,shipped,1107
7,unavailable,609


In [ ]:
reference_date = (
    df["order_purchase_timestamp"].max()
    + pd.Timedelta(days=1)
)

rfm = (
    df.groupby("customer_unique_id")
    .agg(
        Recency=(
            "order_purchase_timestamp",
            lambda x: (
                reference_date - x.max()
            ).days
        ),
        Frequency=(
            "order_id",
            "nunique"
        ),
        Monetary=(
            "revenue",
            "sum"
        )
    )
    .reset_index()
)

rfm.head()

,customer_unique_id,Recency,Frequency,Monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,161,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,164,1,27.19
2,0000f46a3911fa3c0805444483337064,586,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,370,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,337,1,196.89


In [ ]:
rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first"),
    4,
    labels=[4, 3, 2, 1]
).astype(int)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    4,
    labels=[1, 2, 3, 4]
).astype(int)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    4,
    labels=[1, 2, 3, 4]
).astype(int)

rfm["RFM_Score"] = (
    rfm["R_Score"]
    +
    rfm["F_Score"]
    +
    rfm["M_Score"]
)

def segment_customer(score):

    if score >= 10:
        return "Champions"

    elif score >= 8:
        return "Loyal Customers"

    elif score >= 6:
        return "Potential Loyalists"

    elif score >= 4:
        return "Need Attention"

    else:
        return "At Risk"

rfm["Segment"] = (
    rfm["RFM_Score"]
    .apply(segment_customer)
)

display(rfm.head(20))

,customer_unique_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
0,0000366f3b9a7992bf8c76cfdf3221e2,161,1,141.90,4,1,3,8,Loyal Customers
1,0000b849f77a49e4a4ce2b2a4ca5be3f,164,1,27.19,4,1,1,6,Potential Loyalists
2,0000f46a3911fa3c0805444483337064,586,1,86.22,1,1,2,4,Need Attention
3,0000f6ccb0745a6a4b88665a16c9f078,370,1,43.62,2,1,1,4,Need Attention
4,0004aac84e0df4da2b147fca70cf8255,337,1,196.89,2,1,4,7,Potential Loyalists
5,0004bd2a26a76fe21f786e4fbd80607f,195,1,166.98,3,1,3,7,Potential Loyalists
6,00050ab1314c0e55a6ca13cf7181fecf,181,1,35.38,3,1,1,5,Need Attention
7,00053a61a98854899e70ed204dd4bafe,232,1,419.18,3,1,4,8,Loyal Customers
8,0005e1862207bf6ccc02e4228effd9a0,592,1,150.12,1,1,3,5,Need Attention
9,0005ef4cd20d2893f0d9fbd94d3c0d97,220,1,129.76,3,1,3,7,Potential Loyalists


In [ ]:
segment_summary = (
    rfm["Segment"]
    .value_counts()
    .reset_index()
)

segment_summary.columns = [
    "Segment",
    "Customers"
]

fig = px.bar(
    segment_summary,
    x="Segment",
    y="Customers",
    title="Customer Segmentation - RFM"
)

fig.show()

In [ ]:
best_category = (
    category_analysis
    .iloc[0]["product_category_name_english"]
)

best_category_revenue = (
    category_analysis
    .iloc[0]["revenue"]
)

best_state = (
    state_analysis
    .iloc[0]["customer_state"]
)

best_state_revenue = (
    state_analysis
    .iloc[0]["revenue"]
)

best_month = (
    monthly_sales
    .sort_values(
        "revenue",
        ascending=False
    )
    .iloc[0]["year_month"]
)

best_month_revenue = (
    monthly_sales
    .sort_values(
        "revenue",
        ascending=False
    )
    .iloc[0]["revenue"]
)

repeat_percentage = (
    (
        customer_analysis["customer_type"]
        == "Repeat Customer"
    ).mean()
    * 100
)

print("=" * 70)
print("BUSINESS INSIGHTS")
print("=" * 70)

print(
    f"1. Kategori dengan revenue tertinggi adalah "
    f"{best_category}."
)

print(
    f"   Revenue kategori tersebut: "
    f"R$ {best_category_revenue:,.2f}"
)

print()

print(
    f"2. State dengan revenue terbesar adalah "
    f"{best_state}."
)

print(
    f"   Revenue state tersebut: "
    f"R$ {best_state_revenue:,.2f}"
)

print()

print(
    f"3. Bulan dengan revenue tertinggi adalah "
    f"{best_month}."
)

print(
    f"   Revenue bulan tersebut: "
    f"R$ {best_month_revenue:,.2f}"
)

print()

print(
    f"4. Persentase repeat customer sekitar "
    f"{repeat_percentage:.2f}%."
)

print()

print(
    f"5. Rata-rata rating pelanggan adalah "
    f"{average_rating:.2f}/5."
)

BUSINESS INSIGHTS
1. Kategori dengan revenue tertinggi adalah health_beauty.
   Revenue kategori tersebut: R$ 1,441,248.07

2. State dengan revenue terbesar adalah SP.
   Revenue state tersebut: R$ 5,921,678.12

3. Bulan dengan revenue tertinggi adalah 2017-11.
   Revenue bulan tersebut: R$ 1,179,143.77

4. Persentase repeat customer sekitar 3.12%.

5. Rata-rata rating pelanggan adalah 4.02/5.


In [ ]:
df.to_csv(
    "olist_final_clean.csv",
    index=False
)

rfm.to_csv(
    "customer_rfm.csv",
    index=False
)

print("✅ Dataset final berhasil disimpan!")

✅ Dataset final berhasil disimpan!


In [ ]:
import os

print(os.listdir())

['.config', 'olist_data', 'olist_final_clean.csv', 'customer_rfm.csv', 'archive (23).zip', 'sample_data']


In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go



st.set_page_config(
    page_title="E-Commerce Analytics | Ilsan Hidayatuloh",
    page_icon="🛒",
    layout="wide",
    initial_sidebar_state="expanded"
)


st.markdown("""
<style>

    .stApp {
        background-color: #f6f8fb;
    }

    .block-container {
        padding-top: 2rem;
        padding-bottom: 2rem;
        max-width: 1500px;
    }

    .hero {
        padding: 28px;
        border-radius: 20px;
        background: linear-gradient(
            135deg,
            #111827 0%,
            #1f2937 100%
        );
        color: white;
        margin-bottom: 25px;
        box-shadow: 0px 10px 30px rgba(0,0,0,0.12);
    }

    .hero h1 {
        font-size: 38px;
        margin-bottom: 5px;
        font-weight: 800;
    }

    .hero p {
        font-size: 16px;
        opacity: 0.85;
    }

    .author {
        font-size: 14px;
        margin-top: 15px;
        opacity: 0.7;
    }

    .section-title {
        font-size: 24px;
        font-weight: 750;
        margin-top: 15px;
        margin-bottom: 15px;
    }

    [data-testid="stMetric"] {
        background-color: white;
        border-radius: 16px;
        padding: 18px;
        box-shadow: 0px 4px 15px rgba(0,0,0,0.06);
        border: 1px solid #eef0f4;
    }

    [data-testid="stSidebar"] {
        background-color: #ffffff;
    }

    .insight-box {
        padding: 20px;
        border-radius: 15px;
        background-color: white;
        border-left: 5px solid #111827;
        box-shadow: 0px 4px 15px rgba(0,0,0,0.05);
        margin-bottom: 12px;
    }

</style>
""", unsafe_allow_html=True)



@st.cache_data
def load_data():

    data = pd.read_csv(
        "olist_final_clean.csv"
    )

    data["order_purchase_timestamp"] = pd.to_datetime(
        data["order_purchase_timestamp"],
        errors="coerce"
    )

    data["order_estimated_delivery_date"] = pd.to_datetime(
        data["order_estimated_delivery_date"],
        errors="coerce"
    )

    return data


@st.cache_data
def load_rfm():

    rfm_data = pd.read_csv(
        "customer_rfm.csv"
    )

    return rfm_data


df = load_data()
rfm = load_rfm()


st.markdown("""
<div class="hero">

<h1>🛒 E-Commerce Sales Analytics</h1>

<p>
Interactive Dashboard untuk menganalisis performa penjualan,
produk, customer, dan kepuasan pelanggan.
</p>

<div class="author">
Created by <b>Ilsan Hidayatuloh</b> · Data Science Final Project
</div>

</div>
""", unsafe_allow_html=True)



st.sidebar.markdown(
    "# 🛒 E-Commerce Analytics"
)

st.sidebar.markdown(
    "### Dashboard Filter"
)

st.sidebar.caption(
    "Gunakan filter untuk mengeksplorasi data."
)

st.sidebar.divider()



min_date = (
    df["order_purchase_timestamp"]
    .min()
    .date()
)

max_date = (
    df["order_purchase_timestamp"]
    .max()
    .date()
)

date_range = st.sidebar.date_input(
    "📅 Periode",
    value=(min_date, max_date),
    min_value=min_date,
    max_value=max_date
)



categories = sorted(
    df["product_category_name_english"]
    .dropna()
    .unique()
)

selected_categories = st.sidebar.multiselect(
    "📦 Kategori Produk",
    options=categories,
    default=[]
)



states = sorted(
    df["customer_state"]
    .dropna()
    .unique()
)

selected_states = st.sidebar.multiselect(
    "📍 Customer State",
    options=states,
    default=[]
)


statuses = sorted(
    df["order_status"]
    .dropna()
    .unique()
)

selected_statuses = st.sidebar.multiselect(
    "🚚 Order Status",
    options=statuses,
    default=[]
)



filtered_df = df.copy()

if len(date_range) == 2:

    start_date = pd.Timestamp(
        date_range[0]
    )

    end_date = (
        pd.Timestamp(date_range[1])
        +
        pd.Timedelta(days=1)
    )

    filtered_df = filtered_df[
        (
            filtered_df[
                "order_purchase_timestamp"
            ] >= start_date
        )
        &
        (
            filtered_df[
                "order_purchase_timestamp"
            ] < end_date
        )
    ]


if selected_categories:

    filtered_df = filtered_df[
        filtered_df[
            "product_category_name_english"
        ].isin(selected_categories)
    ]


if selected_states:

    filtered_df = filtered_df[
        filtered_df[
            "customer_state"
        ].isin(selected_states)
    ]


if selected_statuses:

    filtered_df = filtered_df[
        filtered_df[
            "order_status"
        ].isin(selected_statuses)
    ]



total_revenue = (
    filtered_df["revenue"].sum()
)

total_orders = (
    filtered_df["order_id"].nunique()
)

total_customers = (
    filtered_df["customer_unique_id"].nunique()
)

total_products = (
    filtered_df["product_id"].nunique()
)

aov = (
    total_revenue / total_orders
    if total_orders > 0
    else 0
)

rating = (
    filtered_df["review_score"].mean()
    if len(filtered_df) > 0
    else 0
)

st.markdown(
    '<div class="section-title">📊 Executive Overview</div>',
    unsafe_allow_html=True
)

c1, c2, c3, c4, c5 = st.columns(5)

c1.metric(
    "💰 Revenue",
    f"R$ {total_revenue:,.0f}"
)

c2.metric(
    "🛒 Orders",
    f"{total_orders:,}"
)

c3.metric(
    "👥 Customers",
    f"{total_customers:,}"
)

c4.metric(
    "💳 Avg Order Value",
    f"R$ {aov:,.0f}"
)

c5.metric(
    "⭐ Avg Rating",
    f"{rating:.2f}/5"
)

st.divider()


overview_tab, sales_tab, product_tab, customer_tab, insight_tab = st.tabs(
    [
        "📊 Overview",
        "📈 Sales",
        "📦 Product",
        "👥 Customer",
        "💡 Insights"
    ]
)



with overview_tab:

    st.markdown(
        '<div class="section-title">📈 Performance Overview</div>',
        unsafe_allow_html=True
    )

    monthly = (
        filtered_df
        .assign(
            year_month=lambda x:
            x[
                "order_purchase_timestamp"
            ]
            .dt.to_period("M")
            .astype(str)
        )
        .groupby("year_month")
        .agg(
            revenue=("revenue", "sum"),
            orders=("order_id", "nunique")
        )
        .reset_index()
    )

    fig = px.line(
        monthly,
        x="year_month",
        y="revenue",
        markers=True,
        title="Monthly Revenue Trend"
    )

    fig.update_layout(
        template="plotly_white",
        hovermode="x unified"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )

    col1, col2 = st.columns(2)

    with col1:

        category = (
            filtered_df
            .groupby(
                "product_category_name_english"
            )
            .agg(
                revenue=("revenue", "sum")
            )
            .sort_values(
                "revenue",
                ascending=False
            )
            .head(10)
            .reset_index()
        )

        fig = px.bar(
            category.sort_values("revenue"),
            x="revenue",
            y="product_category_name_english",
            orientation="h",
            title="Top 10 Product Categories"
        )

        fig.update_layout(
            template="plotly_white"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

    with col2:

        states_chart = (
            filtered_df
            .groupby("customer_state")
            .agg(
                revenue=("revenue", "sum")
            )
            .sort_values(
                "revenue",
                ascending=False
            )
            .head(10)
            .reset_index()
        )

        fig = px.bar(
            states_chart,
            x="customer_state",
            y="revenue",
            title="Top 10 States by Revenue"
        )

        fig.update_layout(
            template="plotly_white"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )


with sales_tab:

    st.markdown(
        '<div class="section-title">📈 Sales Analytics</div>',
        unsafe_allow_html=True
    )

    monthly = (
        filtered_df
        .assign(
            year_month=lambda x:
            x[
                "order_purchase_timestamp"
            ]
            .dt.to_period("M")
            .astype(str)
        )
        .groupby("year_month")
        .agg(
            revenue=("revenue", "sum"),
            orders=("order_id", "nunique")
        )
        .reset_index()
    )

    col1, col2 = st.columns(2)

    with col1:

        fig = px.line(
            monthly,
            x="year_month",
            y="revenue",
            markers=True,
            title="Revenue Trend"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

    with col2:

        fig = px.bar(
            monthly,
            x="year_month",
            y="orders",
            title="Orders per Month"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

    st.subheader(
        "📅 Sales Data"
    )

    st.dataframe(
        monthly,
        use_container_width=True
    )


with product_tab:

    st.markdown(
        '<div class="section-title">📦 Product Analytics</div>',
        unsafe_allow_html=True
    )

    product_analysis = (
        filtered_df
        .groupby(
            "product_category_name_english"
        )
        .agg(
            orders=("order_id", "nunique"),
            quantity=("order_item_id", "count"),
            revenue=("revenue", "sum")
        )
        .sort_values(
            "revenue",
            ascending=False
        )
        .reset_index()
    )

    col1, col2 = st.columns(2)

    with col1:

        fig = px.bar(
            product_analysis
            .head(15)
            .sort_values("revenue"),
            x="revenue",
            y="product_category_name_english",
            orientation="h",
            title="Top Product Categories"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

    with col2:

        fig = px.treemap(
            product_analysis.head(20),
            path=[
                "product_category_name_english"
            ],
            values="revenue",
            title="Product Revenue Treemap"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

    st.subheader(
        "📋 Product Performance Table"
    )

    st.dataframe(
        product_analysis,
        use_container_width=True,
        height=450
    )

with customer_tab:

    st.markdown(
        '<div class="section-title">👥 Customer Analytics</div>',
        unsafe_allow_html=True
    )

    customer_analysis = (
        filtered_df
        .groupby("customer_unique_id")
        .agg(
            orders=("order_id", "nunique"),
            spending=("revenue", "sum")
        )
        .reset_index()
    )

    customer_analysis["customer_type"] = np.where(
        customer_analysis["orders"] > 1,
        "Repeat Customer",
        "One-Time Customer"
    )

    repeat_customers = (
        customer_analysis[
            customer_analysis["orders"] > 1
        ].shape[0]
    )

    repeat_percentage = (
        repeat_customers
        /
        len(customer_analysis)
        *
        100
        if len(customer_analysis) > 0
        else 0
    )

    c1, c2 = st.columns(2)

    c1.metric(
        "Repeat Customers",
        f"{repeat_customers:,}"
    )

    c2.metric(
        "Repeat Customer %",
        f"{repeat_percentage:.2f}%"
    )

    customer_type = (
        customer_analysis[
            "customer_type"
        ]
        .value_counts()
        .reset_index()
    )

    customer_type.columns = [
        "customer_type",
        "customers"
    ]

    col1, col2 = st.columns(2)

    with col1:

        fig = px.pie(
            customer_type,
            names="customer_type",
            values="customers",
            hole=0.55,
            title="Customer Type"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

    with col2:

        top_customers = (
            customer_analysis
            .sort_values(
                "spending",
                ascending=False
            )
            .head(10)
        )

        fig = px.bar(
            top_customers,
            x="spending",
            y="customer_unique_id",
            orientation="h",
            title="Top 10 Customers by Spending"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

    st.subheader(
        "🎯 RFM Customer Segmentation"
    )

    rfm_display = (
        rfm["Segment"]
        .value_counts()
        .reset_index()
    )

    rfm_display.columns = [
        "Segment",
        "Customers"
    ]

    fig = px.bar(
        rfm_display,
        x="Segment",
        y="Customers",
        title="Customer Segments"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )



with insight_tab:

    st.markdown(
        '<div class="section-title">💡 Business Insights & Recommendations</div>',
        unsafe_allow_html=True
    )

    if len(filtered_df) > 0:

        top_category = (
            filtered_df
            .groupby(
                "product_category_name_english"
            )["revenue"]
            .sum()
            .idxmax()
        )

        top_state = (
            filtered_df
            .groupby(
                "customer_state"
            )["revenue"]
            .sum()
            .idxmax()
        )

        top_month = (
            filtered_df
            .assign(
                month=lambda x:
                x[
                    "order_purchase_timestamp"
                ]
                .dt.to_period("M")
                .astype(str)
            )
            .groupby("month")["revenue"]
            .sum()
            .idxmax()
        )

        avg_delivery = (
            filtered_df[
                "delivery_days"
            ]
            .dropna()
            .mean()
        )

        avg_review = (
            filtered_df[
                "review_score"
            ]
            .mean()
        )

        st.markdown(
            f"""
            <div class="insight-box">
            <b>📦 Product Insight</b><br>
            Kategori dengan kontribusi revenue terbesar adalah
            <b>{top_category}</b>.
            </div>
            """,
            unsafe_allow_html=True
        )

        st.markdown(
            f"""
            <div class="insight-box">
            <b>📍 Geographic Insight</b><br>
            State dengan revenue terbesar adalah
            <b>{top_state}</b>.
            </div>
            """,
            unsafe_allow_html=True
        )

        st.markdown(
            f"""
            <div class="insight-box">
            <b>📈 Sales Insight</b><br>
            Periode dengan revenue tertinggi adalah
            <b>{top_month}</b>.
            </div>
            """,
            unsafe_allow_html=True
        )

        st.markdown(
            f"""
            <div class="insight-box">
            <b>🚚 Delivery Insight</b><br>
            Rata-rata waktu pengiriman sekitar
            <b>{avg_delivery:.1f} hari</b>.
            </div>
            """,
            unsafe_allow_html=True
        )

        st.markdown(
            f"""
            <div class="insight-box">
            <b>⭐ Customer Satisfaction</b><br>
            Rata-rata rating customer adalah
            <b>{avg_review:.2f}/5</b>.
            </div>
            """,
            unsafe_allow_html=True
        )

        st.subheader(
            "🚀 Business Recommendations"
        )

        st.markdown(
            """
            **1. Fokus pada kategori dengan revenue tinggi**
            Perusahaan dapat mempertahankan ketersediaan produk
            pada kategori yang memberikan kontribusi revenue terbesar.

            **2. Tingkatkan customer retention**
            Program loyalty, voucher, atau personalized promotion
            dapat digunakan untuk meningkatkan jumlah repeat customer.

            **3. Optimalkan strategi berdasarkan wilayah**
            Wilayah dengan kontribusi penjualan tinggi dapat menjadi
            prioritas untuk campaign marketing.

            **4. Perhatikan pengalaman pengiriman**
            Waktu pengiriman memiliki potensi memengaruhi kepuasan
            pelanggan sehingga proses logistik perlu terus dievaluasi.

            **5. Gunakan customer segmentation**
            Customer dengan segmentasi RFM yang baik dapat diberikan
            program loyalty, sedangkan customer yang mulai tidak aktif
            dapat ditargetkan dengan campaign reactivation.
            """
        )



st.divider()

st.subheader(
    "📥 Download Data"
)

csv_data = filtered_df.to_csv(
    index=False
).encode("utf-8")

st.download_button(
    label="⬇️ Download Filtered Dataset",
    data=csv_data,
    file_name="ecommerce_filtered_data.csv",
    mime="text/csv"
)



st.divider()

st.markdown(
    """
    <center>

    <b>E-Commerce Sales Analytics Dashboard</b><br>

    Brazilian E-Commerce Public Dataset by Olist<br><br>

    <b>Created by Ilsan Hidayatuloh</b><br>

    Data Science Final Project

    </center>
    """,
    unsafe_allow_html=True
)

Writing app.py


In [ ]:
import os

print(os.path.exists("app.py"))
print(os.path.exists("olist_final_clean.csv"))
print(os.path.exists("customer_rfm.csv"))

True
True
True


In [ ]:
!ls -lh app.py olist_final_clean.csv customer_rfm.csv

-rw-r--r-- 1 root root  20K Aug 20 04:30 app.py
-rw-r--r-- 1 root root 6.6M Aug 20 04:30 customer_rfm.csv
-rw-r--r-- 1 root root  55M Aug 20 04:30 olist_final_clean.csv


In [ ]:
!pip install -q pyngrok

In [ ]:
from pyngrok import ngrok

ngrok.kill()

ngrok.set_auth_token(
    "3IAHcPsi6E7JThJ2BXnwpqFoBkP_nH3DAUFEBKYK27afN9nf"
)

print(" Token Ngrok berhasil disimpan.")

✅ Token Ngrok berhasil disimpan.


In [ ]:
!ps aux | grep streamlit

root        9907  6.5  0.5 524552 71544 ?        Sl   04:32   0:03 /usr/bin/python3 /usr/local/bin/streamlit run app.py --server.port 8501 --server.address 0.0.0.0
root       10143  0.0  0.0   7372  3472 ?        S    04:33   0:00 /bin/bash -c ps aux | grep streamlit
root       10145  0.0  0.0   6480  2420 ?        S    04:33   0:00 grep streamlit


In [ ]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8501)

print("=" * 60)
print(" DASHBOARD SIAP!")
print("=" * 60)
print(public_url)

🎉 DASHBOARD SIAP!
NgrokTunnel: "https://handler-powdering-anyplace.ngrok-free.dev" -> "http://localhost:8501"
